# Lab 21-05 · Mixers

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import mixer

db20 = lambda x: 20 * math.log10(x)


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · Spectrum

*Predict* where a 2.45 GHz and a 7.25 GHz tone land when mixed with a 2.40 GHz square wave.

In [2]:
fs, n = 64e9, 1 << 18
t = np.arange(n) / fs
x = np.cos(2 * np.pi * 2.45e9 * t) + np.cos(2 * np.pi * 7.25e9 * t)
y = x * np.sign(np.cos(2 * np.pi * 2.40e9 * t))
f = np.fft.rfftfreq(n, 1 / fs)
sp = np.abs(np.fft.rfft(y * np.hanning(n)))
sp /= sp.max()
plt.figure(); plt.semilogy(f / 1e6, sp); plt.xlim(0, 200); plt.ylim(1e-4, 2)
plt.xlabel("frequency (MHz)"); plt.ylabel("relative amplitude"); plt.show()
print(f"50 MHz line holds both tones: {sp[np.argmin(abs(f - 50e6))]:.2f} of the strongest component")
print(f"3rd-harmonic response is {mixer.conversion_gain_db(1) - mixer.conversion_gain_db(3):.1f} dB below the fundamental")

50 MHz line holds both tones: 0.36 of the strongest component
3rd-harmonic response is 9.5 dB below the fundamental


## Part 2 · ngspice

*Predict* the four measured gains.

In [3]:
s = ngspice("mixer.cir")
print({k: round(s[k], 3) for k in ("gca", "gcb", "gcc", "gnp")})
print("theory:", round(mixer.conversion_gain_db(1), 3), round(mixer.conversion_gain_db(3), 3),
      round(20 * math.log10(2 / math.pi * 0.02 * 500), 3), round(20 * math.log10(((4 / math.pi) * math.sin(math.pi / 4)) ** 2), 3))

{'gca': -3.922, 'gcb': -13.465, 'gcc': 16.078, 'gnp': -1.824}
theory: -3.922 -13.465 16.078 -1.824


## Part 3 · Noise folding

*Predict* the SSB noise figure of a noiseless square-wave mixer with no filter, and with the image removed.

In [4]:
rng = np.random.default_rng(2105)
ns, f_lo, k = 1 << 16, 256, 60
tt = np.arange(ns) / ns
sq = np.sign(np.cos(2 * np.pi * f_lo * tt))


def ratio(remove_image=False, trials=20):
    out = ref = 0.0
    for _ in range(trials):
        spec = np.fft.rfft(rng.normal(0, 1, ns))
        full = spec.copy()
        if remove_image:
            full[f_lo - k:f_lo] = 0
        out += np.sum(abs(np.fft.rfft(np.fft.irfft(full, ns) * sq)[1:k + 1]) ** 2)
        only = np.zeros_like(spec)
        only[f_lo + 1:f_lo + k + 1] = spec[f_lo + 1:f_lo + k + 1]
        ref += np.sum(abs(np.fft.rfft(np.fft.irfft(only, ns) * sq)[1:k + 1]) ** 2)
    return 10 * math.log10(out / ref)


print(f"unfiltered: {ratio():.2f} dB (theory {10 * math.log10(math.pi ** 2 / 4):.2f}); "
      f"image removed: {ratio(True):.2f} dB (theory 1.67)")

unfiltered: 3.83 dB (theory 3.92); image removed: 1.60 dB (theory 1.67)


## Part 4 · Image and harmonic rejection

*Predict* the IRR at 1 % and 1°, and the harmonic-reject LO's 3rd response for 1 % weight error.

In [5]:
eps = np.linspace(0.001, 0.05, 100)
th = np.linspace(0.05, 5, 100)
E, TH = np.meshgrid(eps, th)
irr = np.vectorize(mixer.irr_db)(E, TH)
plt.figure(); cs = plt.contour(E * 100, TH, irr, levels=[30, 40, 50]); plt.clabel(cs)
plt.xlabel("gain error (%)"); plt.ylabel("phase error (deg)"); plt.show()
print(f"IRR at 1 %, 1 deg: {mixer.irr_db(0.01, 1):.1f} dB")
for err, (ph_shift, w_scale) in (("1 % weight", (0.0, 1.01)), ("1 deg phase", (1.0, 1.0))):
    w = [(-45.0, 1.0), (0.0, math.sqrt(2) * w_scale), (45.0 + ph_shift, 1.0)]
    h = mixer.composite_lo_harmonics(w, nmax=5, npts=1 << 16)
    print(f"{err}: 3rd {db20(h[3]):.1f} dB, 5th {db20(h[5]):.1f} dB")

IRR at 1 %, 1 deg: 40.0 dB
1 % weight: 3rd -55.6 dB, 5th -60.0 dB
1 deg phase: 3rd -44.2 dB, 5th -44.2 dB


## Part 5 · N-path filter

*Predict* the centre loss and the response at ±BW/2 of Example 21-05.8's filter.

In [6]:
def npath(n_paths, r, c, f_lo, f_in, periods=3000, steps=100, last=1000):
    tt = 1 / f_lo
    dt = tt / steps
    t = np.arange(periods * steps) * dt
    vs = np.cos(2 * np.pi * f_in * t)
    k_of = ((np.arange(periods * steps) % steps) * n_paths // steps).astype(int)
    a = math.exp(-dt / (r * c))
    caps = np.zeros(n_paths)
    out = np.empty_like(vs)
    for i in range(len(vs)):
        kk = k_of[i]
        caps[kk] = vs[i] + (caps[kk] - vs[i]) * a
        out[i] = caps[kk]
    m = slice(len(vs) - last * steps, len(vs))
    return abs(2 * np.mean(out[m] * np.exp(-2j * np.pi * f_in * t[m])))


bw = 1 / (math.pi * 4 * 50 * 200e-12)
c0 = npath(4, 50, 200e-12, 1e9, 1e9)
print(f"centre {db20(c0):.2f} dB; +BW/2 {db20(npath(4, 50, 200e-12, 1e9, 1e9 + bw / 2) / c0):.2f} dB; "
      f"-BW/2 {db20(npath(4, 50, 200e-12, 1e9, 1e9 - bw / 2) / c0):.2f} dB (BW = {bw / 1e6:.2f} MHz)")

centre -1.82 dB; +BW/2 -2.96 dB; -BW/2 -3.06 dB (BW = 7.96 MHz)
